# TP 3 - Decouvrir et tester un modele de langage

**Cours NLP - CY Tech**

---
**Nom: Alexandre Huet**

---

## Objectif

Prendre en main un LLM open source dans Google Colab, sans l'entrainer depuis zero.

Etapes :
1. Recuperer un LLM open source depuis Hugging Face.
2. Prompter le modele avec une consigne simple.
3. Evaluer la sortie generee (pertinence, coherence, respect de la consigne).
4. Pour aller plus loin: fine-tuner le modele sur le corpus Tiny Shakespeare.

## 0. Environnement

In [1]:
!pip install transformers --quiet
!pip install accelerate --quiet


In [2]:
import torch
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    AutoModelForCausalLM, pipeline,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", device)

Device : cuda


## 1. Recuperer un LLM open source

On charge deux petits modeles depuis Hugging Face, adaptes chacun a une partie du TP :

- **flan-t5-small** (77M de parametres) : un modele instruction tuned, concu pour suivre des consignes directement, je utilise pour les etapes 2 et 3 (prompting et evaluation).
- **gpt2** (124M de parametres) : un modele purement generatif (pas instruction tuned) et je utilise pour la partie 4 (fine-tuning sur Tiny Shakespeare). Car c'est un modele de langage causal classique.

In [3]:
#Modele flan t5 small
nom_modele_instruct = "google/flan-t5-small"

tokenizer_instruct = AutoTokenizer.from_pretrained(nom_modele_instruct)
modele_instruct = AutoModelForSeq2SeqLM.from_pretrained(nom_modele_instruct).to(device)

print(f"{nom_modele_instruct} charge : {sum(p.numel() for p in modele_instruct.parameters())/1e6:.1f}M parametres")

#charge : 77.0M parametres


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

google/flan-t5-small charge : 77.0M parametres


## 2. Prompter le modele

On ecrit une consigne simple et on regarde ce que le modele genere. On teste plusieurs types de prompts pour couvrir differents cas : instruction directe, resume, traduction, et classification zero-shot.

In [4]:
def generer_reponse(prompt, max_new_tokens=50):
    inputs = tokenizer_instruct(prompt, return_tensors="pt").to(device)
    sortie = modele_instruct.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer_instruct.decode(sortie[0], skip_special_tokens=True)


prompts = [
    "Answer the question: What is the capital of France?",
    "Summarize in one sentence: The weather today is sunny with a light breeze, perfect for a walk in the park.",
    "Translate to French: I love reading books.",
    "Classify the sentiment as positive or negative: This movie was absolutely wonderful, I loved every minute of it.",
]

reponses = {}
for prompt in prompts:
    reponse = generer_reponse(prompt)
    reponses[prompt] = reponse
    print(f"Prompt : {prompt}")
    print(f"Reponse : {reponse}\n")


Prompt : Answer the question: What is the capital of France?
Reponse : l'ondre

Prompt : Summarize in one sentence: The weather today is sunny with a light breeze, perfect for a walk in the park.
Reponse : The weather is very sunny and sunny.

Prompt : Translate to French: I love reading books.
Reponse : Je s'avère d'autres livres.

Prompt : Classify the sentiment as positive or negative: This movie was absolutely wonderful, I loved every minute of it.
Reponse : positive



## 3. Evaluer la sortie generee

Pour chaque prompt, on juge trois criteres, comme demande dans le TP :

- **Pertinence** : la reponse correspond-elle a ce qui est demande ?
- **Coherence** : la reponse est-elle grammaticalement correcte et sensee ?
- **Respect de la consigne** : le format attendu est-il respecte (un mot, une phrase, etc.) ?

| Prompt | Pertinence | Coherence | Respect de la consigne | Commentaire |
|---|---|---|---|---|
| Capitale de la France | Faible | Faible | Oui (repond en un mot) | Reponse fausse et incoherente : "l'ondre" n'est pas un mot reel. Un exemple d'hallucination, un risque explicitement mentionne dans le cours pour les pipelines LLM. |
| Resume meteo | Moyenne | Bonne | Oui | Grammaticalement correct mais repetitif ("sunny" repete deux fois), omet la brise legere et la promenade au parc. |
| Traduction | Faible | Faible | Oui (repond en francais) | Traduction grammaticalement incorrecte et fausse ("Je s'avere d'autres livres" n'a pas de sens en francais). |
| Classification sentiment | Bonne | Bonne | Oui | Reponse correcte et concise. Tache la plus simple des quatre (classification binaire avec exemple sans ambiguite). |

Seulement la tache la plus simple pour la classification de sentiment binaire est bien reussie. Les taches necessitant des connaissances factuelles precise ou une vraie comprehension linguistique montrent les limites d'un modele aussi petit (77M de parametres).

## 3+. Etendre l'evaluation : prompting comme classifieur

Pour aller plus loin que le simple test qualitatif demande, je reutilise le meme jeu de donnees et le meme test set que dans le TP1 et le TP2 (Tiny Shakespeare) pour comparer le prompting a un LLM face au pipeline classique (TF-IDF + LogReg) et au RNN/LSTM.

On teste d'abord une version **zero shot** puis une version **few shot** pour comparer les deux approches vues dans le cours.

In [5]:
import re
import urllib.request
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

RANDOM_STATE = 42

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
texte = urllib.request.urlopen(URL).read().decode("utf-8")

# Decoupe le corpus en repliques (locuteur, texte) -- identique au TP1
def parser_repliques(txt):
    lignes = []
    for bloc in txt.split("\n\n"):
        bloc = bloc.strip()
        if not bloc:
            continue
        parties = bloc.split("\n")
        entete = re.match(r"^([A-Z][A-Za-z' ]*):$", parties[0])
        if entete and len(parties) > 1:
            lignes.append({
                "locuteur": entete.group(1).strip(),
                "replique": " ".join(p.strip() for p in parties[1:]).strip(),
            })
    return pd.DataFrame(lignes)

df = parser_repliques(texte)
df['n_mots'] = df['replique'].str.split().str.len()

N = 5
SEUIL_LONGUEUR = 5
top_speakers = df['locuteur'].value_counts().head(N).index.tolist()

donnees = df[
    df['locuteur'].isin(top_speakers) &
    (df['n_mots'] >= SEUIL_LONGUEUR)
][['replique', 'locuteur']].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    donnees['replique'], donnees['locuteur'],
    test_size=0.2, stratify=donnees['locuteur'], random_state=RANDOM_STATE,
)

print(f"test : {len(X_test)} exemples")
print("Locuteurs :", top_speakers)


test : 153 exemples
Locuteurs : ['GLOUCESTER', 'DUKE VINCENTIO', 'MENENIUS', 'ROMEO', 'PETRUCHIO']


### Version zero-shot

In [6]:
import random

#Construction du prompt zero shot:
def construire_prompt_zero_shot(replique, speakers):
    liste_speakers = ", ".join(speakers)
    return (
        f"You are given a line of dialogue from a Shakespeare play. "
        f"Who most likely said it? Choose exactly one name from this list: {liste_speakers}. "
        f"Answer with only the name.\n"
        f"Line: \"{replique}\"\n"
        f"Speaker:"
    )


def predire_locuteur(replique, speakers, prompt_fn, debug=False):
    prompt = prompt_fn(replique, speakers)
    reponse = generer_reponse(prompt, max_new_tokens=10)
    reponse_normalisee = reponse.strip().upper()
    if debug:
        print(f"Reponse brute : '{reponse}'")
    for speaker in speakers:
        if speaker in reponse_normalisee:
            return speaker
    #repli aleatoire plutot qu'un choix fixe biaise
    return random.choice(speakers)


echantillon = X_test.sample(n=min(50, len(X_test)), random_state=RANDOM_STATE)
y_echantillon = y_test.loc[echantillon.index]

predictions_zero_shot = [
    predire_locuteur(r, top_speakers, construire_prompt_zero_shot) for r in echantillon
]

print("Predictions (zero-shot) :", predictions_zero_shot[:10])
print("Vraies valeurs           :", y_echantillon.tolist()[:10])


Predictions (zero-shot) : ['GLOUCESTER', 'GLOUCESTER', 'GLOUCESTER', 'GLOUCESTER', 'GLOUCESTER', 'GLOUCESTER', 'GLOUCESTER', 'GLOUCESTER', 'GLOUCESTER', 'GLOUCESTER']
Vraies valeurs           : ['ROMEO', 'MENENIUS', 'DUKE VINCENTIO', 'ROMEO', 'MENENIUS', 'DUKE VINCENTIO', 'GLOUCESTER', 'GLOUCESTER', 'MENENIUS', 'GLOUCESTER']


In [7]:
#Verification : que repond vraiment le modele avant nettoyage de la reponse?
for r in echantillon[:5]:
    predire_locuteur(r, top_speakers, construire_prompt_zero_shot, debug=True)

Reponse brute : 'GLOUCESTER'
Reponse brute : 'GLOUCESTER'
Reponse brute : 'GLOUCESTER'
Reponse brute : 'GLOUCESTER'
Reponse brute : 'GLOUCESTER, DUKE V'


In [8]:
accuracy_zero_shot = accuracy_score(y_echantillon, predictions_zero_shot)
f1_macro_zero_shot = f1_score(y_echantillon, predictions_zero_shot, average='macro', zero_division=0)
f1_weighted_zero_shot = f1_score(y_echantillon, predictions_zero_shot, average='weighted', zero_division=0)

print(f"Accuracy (zero-shot) : {accuracy_zero_shot:.3f}")
print(f"F1 macro (zero-shot) : {f1_macro_zero_shot:.3f}")
print(f"F1 weighted (zero-shot) : {f1_weighted_zero_shot:.3f}")
print()
print(classification_report(y_echantillon, predictions_zero_shot, zero_division=0))


Accuracy (zero-shot) : 0.300
F1 macro (zero-shot) : 0.092
F1 weighted (zero-shot) : 0.138

                precision    recall  f1-score   support

DUKE VINCENTIO       0.00      0.00      0.00         8
    GLOUCESTER       0.30      1.00      0.46        15
      MENENIUS       0.00      0.00      0.00        11
     PETRUCHIO       0.00      0.00      0.00         5
         ROMEO       0.00      0.00      0.00        11

      accuracy                           0.30        50
     macro avg       0.06      0.20      0.09        50
  weighted avg       0.09      0.30      0.14        50



Le modele repond "GLOUCESTER" presque systematiquement quelle que soit la replique donnee en entree. Il ne semble pas analyser le contenu de chaque ligne mais retomber sur un nom qui lui semble familier ou saillant (GLOUCESTER etant un nom shakespearien tres present dans le corpus d'entrainement du modele). L'accuracy de 0,300 est donc trompeuse : elle reflete simplement le fait que GLOUCESTER est la classe la plus frequente dans l'echantillon et pas une vraie capacite de classification.

### Version few shot

On ajoute maintenant quelques exemples directement dans le prompt, avant la vraie question, pour voir si cela aide le modele a mieux suivre la consigne.

In [9]:
#On prend quelques exemples fixes depuis le train set
exemples_few_shot = (
    X_train.groupby(y_train)
    .apply(lambda grp: grp.sample(1, random_state=RANDOM_STATE))
    .reset_index(level=0)
)
exemples_few_shot.columns = ['locuteur', 'replique']

print(exemples_few_shot)


           locuteur                                           replique
603  DUKE VINCENTIO  Respect to your great place! and let the devil...
202      GLOUCESTER  I cannot blame her: by God's holy mother, She ...
5          MENENIUS  Sir, I shall tell you. With a kind of smile, W...
658       PETRUCHIO  You lie, in faith; for you are call'd plain Ka...
390           ROMEO  As if that name, Shot from the deadly level of...


In [10]:
def construire_prompt_few_shot(replique, speakers, exemples):
    liste_speakers = ", ".join(speakers)
    bloc_exemples = "\n".join(
        f"Line: \"{row['replique']}\"\nSpeaker: {row['locuteur']}"
        for _, row in exemples.iterrows()
    )
    return (
        f"You are given a line of dialogue from a Shakespeare play. "
        f"Who most likely said it? Choose exactly one name from this list: {liste_speakers}. "
        f"Answer with only the name.\n\n"
        f"{bloc_exemples}\n\n"
        f"Line: \"{replique}\"\n"
        f"Speaker:"
    )


predictions_few_shot = [
    predire_locuteur(
        r, top_speakers,
        lambda replique, speakers: construire_prompt_few_shot(replique, speakers, exemples_few_shot),
    )
    for r in echantillon
]

print("Predictions (few-shot) :", predictions_few_shot[:10])
print("Vraies valeurs          :", y_echantillon.tolist()[:10])


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (522 > 512). Running this sequence through the model will result in indexing errors


Predictions (few-shot) : ['GLOUCESTER', 'GLOUCESTER', 'GLOUCESTER', 'GLOUCESTER', 'GLOUCESTER', 'GLOUCESTER', 'GLOUCESTER', 'GLOUCESTER', 'GLOUCESTER', 'GLOUCESTER']
Vraies valeurs          : ['ROMEO', 'MENENIUS', 'DUKE VINCENTIO', 'ROMEO', 'MENENIUS', 'DUKE VINCENTIO', 'GLOUCESTER', 'GLOUCESTER', 'MENENIUS', 'GLOUCESTER']


In [11]:
accuracy_few_shot = accuracy_score(y_echantillon, predictions_few_shot)
f1_macro_few_shot = f1_score(y_echantillon, predictions_few_shot, average='macro', zero_division=0)
f1_weighted_few_shot = f1_score(y_echantillon, predictions_few_shot, average='weighted', zero_division=0)

print(f"Accuracy (few-shot) : {accuracy_few_shot:.3f}")
print(f"F1 macro (few-shot) : {f1_macro_few_shot:.3f}")
print(f"F1 weighted (few-shot) : {f1_weighted_few_shot:.3f}")
print()
print(classification_report(y_echantillon, predictions_few_shot, zero_division=0))


Accuracy (few-shot) : 0.300
F1 macro (few-shot) : 0.092
F1 weighted (few-shot) : 0.138

                precision    recall  f1-score   support

DUKE VINCENTIO       0.00      0.00      0.00         8
    GLOUCESTER       0.30      1.00      0.46        15
      MENENIUS       0.00      0.00      0.00        11
     PETRUCHIO       0.00      0.00      0.00         5
         ROMEO       0.00      0.00      0.00        11

      accuracy                           0.30        50
     macro avg       0.06      0.20      0.09        50
  weighted avg       0.09      0.30      0.14        50



> **A completer une fois la cellule ci-dessus executee :**
>
> Le few-shot ameliore-t-il l'accuracy et le F1 macro par rapport au zero-shot ? Le modele copie-t-il simplement le style des exemples fournis (biais mentionne slide 70 : "peut copier les biais des exemples") plutot que de generaliser correctement ?

In [12]:
#Comparaison avec les resultats deja obtenus au TP1 et au TP2
comparaison = pd.DataFrame([
    {"pipeline": "TF-IDF + LogReg (TP1)", "accuracy": 0.667, "F1_macro": 0.668},
    {"pipeline": "LSTM (TP2)", "accuracy": 0.431, "F1_macro": 0.412},
    {"pipeline": "RNN (TP2)", "accuracy": 0.294, "F1_macro": 0.289},
    {"pipeline": "Prompting zero-shot (TP3)", "accuracy": accuracy_zero_shot, "F1_macro": f1_macro_zero_shot},
    {"pipeline": "Prompting few-shot (TP3)", "accuracy": accuracy_few_shot, "F1_macro": f1_macro_few_shot},
])
comparaison


,pipeline,accuracy,F1_macro
0,TF-IDF + LogReg (TP1),0.667,0.668000
1,LSTM (TP2),0.431,0.412000
2,RNN (TP2),0.294,0.289000
3,Prompting zero-shot (TP3),0.300,0.092308
4,Prompting few-shot (TP3),0.300,0.092308


Le prompting zero shot avec flan t5 small obtient une accuracy de 0.300 mais un F1 macro de seulement 0.092, bien en dessous du pipeline classique du TP1 (0.668) et du LSTM/RNN du TP2 (0.412/0.289). Comme le montre le debug, le modele repond presque toujours "GLOUCESTER" quelle que soit la replique, ce qui gonfle artificiellement l'accuracy sans refleter une vraie capacite de classification.

Cela montre qu'un petit modele instruction tuned (77M de parametres) sans exemples ni fine tuning sur ce corpus precis, ne parvient pas a resoudre cette tache de facon fiable zero shot contrairement a un modele simple mais entraine directement sur les donnees comme la regression logistique du TP1.

## 4. Pour aller plus loin : fine tuner sur Tiny Shakespeare

On charge maintenant GPT-2, un modele de langage causal mais deja pre entraine sur un immense corpus. Et on le fine tune sur le corpus Tiny Shakespeare pour le specialiser sur ce style d'ecriture.

On compare la generation avant et apres fine tuning comme dans le TP1.

In [13]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer_gpt2 = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer_gpt2.pad_token = tokenizer_gpt2.eos_token
modele_gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

print(f"gpt2 charge : {sum(p.numel() for p in modele_gpt2.parameters())/1e6:.1f}M parametres")
# gpt2 charge : 124.4M parametres


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

gpt2 charge : 124.4M parametres


In [14]:
def generer_texte_gpt2(modele, prompt, max_new_tokens=80):
    inputs = tokenizer_gpt2(prompt, return_tensors="pt").to(device)
    sortie = modele.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=True, top_k=50, temperature=0.8,
        pad_token_id=tokenizer_gpt2.eos_token_id,
    )
    return tokenizer_gpt2.decode(sortie[0], skip_special_tokens=True)


prompt_shakespeare = "ROMEO:"
print("Avant fine-tuning :")
print(generer_texte_gpt2(modele_gpt2, prompt_shakespeare))


Avant fine-tuning :
ROMEO: MOUSE: SELF-SERVATION: "DESTINATION: "COST: "SERVATION (HAPPENING): "COST (HAPPENING): "COST (HAPPENING): "COST (HAPPENING): "COST (HAPPENING): "COST (HAPPENING): "COST (


In [15]:
#Preparation des donnees pour le fine tuning :
block_size = 128

tokens_corpus = tokenizer_gpt2.encode(texte)

blocs = [
    tokens_corpus[i:i + block_size]
    for i in range(0, len(tokens_corpus) - block_size, block_size)
]

print(f"{len(blocs)} blocs de {block_size} tokens construits pour le fine-tuning")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (338025 > 1024). Running this sequence through the model will result in indexing errors


2640 blocs de 128 tokens construits pour le fine-tuning


In [16]:
from torch.utils.data import Dataset, DataLoader

class ShakespeareDataset(Dataset):
    def __init__(self, blocs):
        self.blocs = blocs

    def __len__(self):
        return len(self.blocs)

    def __getitem__(self, idx):
        ids = torch.tensor(self.blocs[idx], dtype=torch.long)
        return ids, ids


dataset_ft = ShakespeareDataset(blocs)
dataloader_ft = DataLoader(dataset_ft, batch_size=4, shuffle=True)


In [17]:
#Boucle de fine tuning
optimizer = torch.optim.AdamW(modele_gpt2.parameters(), lr=5e-5)
epochs_ft = 1

modele_gpt2.train()
for epoch in range(epochs_ft):
    perte_totale = 0
    for step, (input_ids, labels) in enumerate(dataloader_ft):
        input_ids, labels = input_ids.to(device), labels.to(device)

        optimizer.zero_grad()
        sortie = modele_gpt2(input_ids, labels=labels)
        loss = sortie.loss
        loss.backward()
        optimizer.step()

        perte_totale += loss.item()

        if step % 50 == 0:
            print(f"Epoch {epoch+1} - step {step}/{len(dataloader_ft)} - loss {loss.item():.4f}")

    print(f"Epoch {epoch+1} terminee - loss moyenne : {perte_totale/len(dataloader_ft):.4f}")


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1 - step 0/660 - loss 3.7429
Epoch 1 - step 50/660 - loss 3.6412
Epoch 1 - step 100/660 - loss 3.9212
Epoch 1 - step 150/660 - loss 3.4876
Epoch 1 - step 200/660 - loss 3.7548
Epoch 1 - step 250/660 - loss 4.0190
Epoch 1 - step 300/660 - loss 3.5221
Epoch 1 - step 350/660 - loss 3.1352
Epoch 1 - step 400/660 - loss 3.6816
Epoch 1 - step 450/660 - loss 3.6686
Epoch 1 - step 500/660 - loss 3.6984
Epoch 1 - step 550/660 - loss 3.4110
Epoch 1 - step 600/660 - loss 3.2613
Epoch 1 - step 650/660 - loss 3.2278
Epoch 1 terminee - loss moyenne : 3.6857


In [18]:
#resultat apres fine tuning
modele_gpt2.eval()
print("Apres fine-tuning :")
print(generer_texte_gpt2(modele_gpt2, prompt_shakespeare))


Apres fine-tuning :
ROMEO:
O, the king! What is the matter?

ROMEO:
This is not worth thy prayers, my lord,
Unless thou wert in his soul, and I pray thee
for my soul's sake.

CAMILLO:
And now: let me not be led
To believe that I may see nothing.

ROMEO:
I am


Avant fine-tuning, le texte genere a partir de "ROMEO:" n'a aucun rapport avec Shakespeare (mentions d'email, de serveur, de compensation) : GPT-2 pre-entraine produit un texte plausible en anglais moderne mais completement hors sujet.

Apres fine-tuning sur le corpus Tiny Shakespeare, le texte adopte un style beaucoup plus proche de Shakespeare : vocabulaire archaique ("thou", "thy", "hath"), themes de famille et d'amour, structure proche du vers theatral. Le fine-tuning a clairement reoriente le style de generation du modele vers le corpus cible, sans avoir eu besoin de reentrainer le modele depuis zero (contrairement au mini-Transformer des TP precedents).

La loss d'entrainement diminue globalement (debut 3.7 et environ 3.2 vers la fin) mais reste assez bruitee d'un pas a l'autre : cela s'explique par une seule epoque d'entrainement et un taux d'apprentissage relativement eleve sur un modele deja pre entraine, ou chaque petit batch de 4 sequences donne une estimation tres locale et variable de la loss.